# Metaeval Full Pipeline Test

This notebook replicates the full dissertation evaluation pipeline using only:
- **metaeval** - for data prep, conversion, variants, judging, and analysis
- **lm-eval** - for model inference

## Pipeline Overview

| Phase | Task | Tool | Command |
|-------|------|------|--------|
| 1 | Download benchmark | metaeval | `metaeval download` |
| 2 | Convert MCQ → OSQ | metaeval | `metaeval convert` |
| 3 | Generate variants | metaeval | `metaeval variants` |
| 4 | Run inference | lm-eval | `lm_eval --model ...` |
| 5 | Judge OSQ responses | metaeval | `metaeval judge` |
| 6 | Analyze results | metaeval | `metaeval analyze` |

In [ ]:
# Setup - ensure we're in the right directory
import os
from pathlib import Path

# Set working directory
WORKDIR = Path("/home/user/dissertation/metaeval-test")
os.chdir(WORKDIR)

# Directory structure
DATA_DIR = WORKDIR / "data"
VARIANTS_DIR = WORKDIR / "variants"
OUTPUT_DIR = WORKDIR / "output"
JUDGED_DIR = WORKDIR / "judged"
ANALYSIS_DIR = WORKDIR / "analysis"

print(f"Working directory: {WORKDIR}")
print(f"Directories: {[d.name for d in WORKDIR.iterdir() if d.is_dir()]}")

---
## Phase 1: Download Benchmark Data

Download the SysEngBench dataset from HuggingFace.

In [ ]:
# Check if data already exists
benchmark_file = DATA_DIR / "ryan-a-bell_SysEngBench.csv"

if benchmark_file.exists():
    print(f"Benchmark already downloaded: {benchmark_file}")
else:
    !metaeval download ryan-a-bell/SysEngBench -o {DATA_DIR}

In [ ]:
# Inspect the downloaded data
import pandas as pd

df = pd.read_csv(benchmark_file)
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nSample question:")
print(df.iloc[0].to_dict())

---
## Phase 2: Convert MCQ to OSQ

Convert multiple-choice questions to open-style questions with rubrics.

**Note:** This requires an OpenAI API key set in the environment.

In [ ]:
# Check for API key
import os

if not os.getenv("OPENAI_API_KEY"):
    print("⚠️  OPENAI_API_KEY not set - skipping conversion")
    print("   Set it with: export OPENAI_API_KEY=your-key")
    SKIP_CONVERSION = True
else:
    print("✓ OPENAI_API_KEY found")
    SKIP_CONVERSION = False

In [ ]:
# Convert MCQ to OSQ (small sample for testing)
osq_file = DATA_DIR / "sysengbench_osq.csv"

if SKIP_CONVERSION:
    print("Skipping conversion - copy existing OSQ data if available")
    # Try to copy from main src directory
    src_osq = Path("/home/user/dissertation/src/phase2_conversion/data/sysengbench_osq_filtered.csv")
    if src_osq.exists():
        import shutil
        shutil.copy(src_osq, osq_file)
        print(f"Copied existing OSQ data from {src_osq}")
elif osq_file.exists():
    print(f"OSQ file already exists: {osq_file}")
else:
    # Convert a small sample (first 50 questions) for testing
    sample_file = DATA_DIR / "sysengbench_sample.csv"
    df.head(50).to_csv(sample_file, index=False)
    
    !metaeval convert {sample_file} -o {osq_file} --threshold 7 --model gpt-4o

In [ ]:
# Inspect OSQ data if available
if osq_file.exists():
    osq_df = pd.read_csv(osq_file)
    print(f"OSQ dataset shape: {osq_df.shape}")
    print(f"\nColumns: {osq_df.columns.tolist()}")
    if len(osq_df) > 0:
        print(f"\nSample OSQ:")
        sample = osq_df.iloc[0]
        print(f"Question: {sample.get('osq_question', sample.get('question', 'N/A'))[:200]}...")
else:
    print("No OSQ file available")

---
## Phase 3: Generate Position Variants

Create A, B, C, D position variants for bias analysis.

In [ ]:
# Generate position variants
variant_files = list(VARIANTS_DIR.glob("*.csv"))

if len(variant_files) >= 4:
    print(f"Variants already exist: {[f.name for f in variant_files]}")
else:
    !metaeval variants {benchmark_file} -o {VARIANTS_DIR}

In [ ]:
# List generated variants
variant_files = sorted(VARIANTS_DIR.glob("*.csv"))
print(f"Generated {len(variant_files)} variant files:")
for vf in variant_files:
    vdf = pd.read_csv(vf)
    print(f"  {vf.name}: {len(vdf)} questions")

---
## Phase 4: Run Model Inference (lm-eval)

This phase uses lm-eval directly. Metaeval provides documentation only.

### Option A: Use existing results from src/
### Option B: Run lm-eval manually

In [ ]:
# Show lm-eval documentation
!metaeval eval --example mcq

In [ ]:
# Option A: Link to existing results
src_output = Path("/home/user/dissertation/src/phase4_inference/output")

if src_output.exists() and not list(OUTPUT_DIR.glob("*")):
    # Create symlink to existing results
    import shutil
    shutil.rmtree(OUTPUT_DIR)  # Remove empty dir
    OUTPUT_DIR.symlink_to(src_output)
    print(f"Linked to existing results: {src_output}")
elif list(OUTPUT_DIR.glob("*")):
    print(f"Output directory already has content")
else:
    print("No existing results found. Run lm-eval manually:")
    print(f"""
# Example: Run with Ollama locally
lm_eval \\
  --model local-chat-completions \\
  --model_args model=llama3.2:3b,base_url=http://localhost:11434/v1/chat/completions \\
  --tasks {VARIANTS_DIR}/ryan-a-bell_SysEngBench_a.csv \\
  --output_path {OUTPUT_DIR} \\
  --log_samples \\
  --batch_size auto
""")

In [ ]:
# List available results
!metaeval results list {OUTPUT_DIR} --format table 2>/dev/null | head -30

In [ ]:
# Summary of results
!metaeval results summary {OUTPUT_DIR}

---
## Phase 5: Judge OSQ Responses

Run LLM-as-a-Judge evaluation on OSQ responses.

**Note:** Requires API access to judge model (Ollama, OpenAI, etc.)

In [ ]:
# Check for OSQ results to judge
!metaeval results list {OUTPUT_DIR} --task osq --format table 2>/dev/null

In [ ]:
# Example: Judge one model's OSQ responses
# Pick the first available OSQ model output
osq_dirs = list((OUTPUT_DIR / "sysengbench-osq").glob("*")) if (OUTPUT_DIR / "sysengbench-osq").exists() else []

if osq_dirs:
    sample_model_dir = osq_dirs[0]
    print(f"Sample model to judge: {sample_model_dir.name}")
    print(f"\nTo judge with Ollama (local):")
    print(f"  metaeval judge {sample_model_dir} --provider ollama --model llama3.3:70b -o {JUDGED_DIR}/judged.jsonl")
    print(f"\nTo judge with OpenAI:")
    print(f"  metaeval judge {sample_model_dir} --provider openai --model gpt-4o -o {JUDGED_DIR}/judged.jsonl")
else:
    print("No OSQ results found to judge")

In [ ]:
# Uncomment to actually run judging (requires API access)
# !metaeval judge {osq_dirs[0]} --provider ollama --model llama3.3:70b -o {JUDGED_DIR}/judged.jsonl

---
## Phase 6: Analyze Results

Run position bias analysis and MCQ vs OSQ comparison.

In [ ]:
# Run position bias analysis
!metaeval analyze bias {OUTPUT_DIR} -o {ANALYSIS_DIR} --format json

In [ ]:
# Check analysis output
bias_file = ANALYSIS_DIR / "bias_results.json"
if bias_file.exists():
    import json
    with open(bias_file) as f:
        bias_results = json.load(f)
    
    print(f"Analyzed {len(bias_results)} models")
    print(f"\nTop 5 by accuracy:")
    sorted_models = sorted(bias_results.items(), key=lambda x: x[1]['overall_accuracy'], reverse=True)
    for model, data in sorted_models[:5]:
        print(f"  {model}: {data['overall_accuracy']:.1%}")

In [ ]:
# MCQ vs OSQ comparison (requires judged results)
judged_files = list(JUDGED_DIR.glob("*.jsonl"))

if judged_files:
    !metaeval analyze compare {OUTPUT_DIR} --judged {JUDGED_DIR} -o {ANALYSIS_DIR} --format json
else:
    print("No judged results available for comparison analysis")
    print(f"Run judging first: metaeval judge <osq_dir> --provider <provider> -o {JUDGED_DIR}/judged.jsonl")

---
## Summary

This notebook demonstrated the full metaeval pipeline:

1. **Download**: `metaeval download ryan-a-bell/SysEngBench`
2. **Convert**: `metaeval convert input.csv -o output.csv`
3. **Variants**: `metaeval variants input.csv -o variants/`
4. **Inference**: `lm_eval --model ... --tasks ... --output_path output/`
5. **Judge**: `metaeval judge output/osq/model/ --provider ollama`
6. **Analyze**: `metaeval analyze bias output/` + `metaeval analyze compare output/ --judged judged/`

In [ ]:
# Final directory listing
print("\n=== Directory Contents ===")
for subdir in [DATA_DIR, VARIANTS_DIR, OUTPUT_DIR, JUDGED_DIR, ANALYSIS_DIR]:
    if subdir.exists():
        files = list(subdir.iterdir())
        print(f"\n{subdir.name}/: {len(files)} items")
        for f in files[:5]:
            print(f"  - {f.name}")
        if len(files) > 5:
            print(f"  ... and {len(files) - 5} more")